# OCR Eval Live Filters Starter

Interactive view for transcript-backed OCR evals with instant filter updates.

Data sources (local lane):
- `.local/eval_cases/ocr_transcript_cases_review.json`
- `.local/eval_cases/ocr_transcript_cases_all.json`
- `.local/eval_reports/ocr_transcript_stability.json`

If data is missing, run:
- `make ocr-data CGPT_EXPORT_ROOT=/abs/path/to/CGPT-DATA-EXPORT`


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.express as px
from IPython.display import HTML, clear_output, display


In [ ]:
def _find_repo_root(start: Path) -> Path:
    probe = start
    for candidate in (probe, *probe.parents):
        if (candidate / '.git').exists():
            return candidate
    return start

ROOT = _find_repo_root(Path.cwd().resolve())
REVIEW_PATH = ROOT / '.local' / 'eval_cases' / 'ocr_transcript_cases_review.json'
ALL_CASES_PATH = ROOT / '.local' / 'eval_cases' / 'ocr_transcript_cases_all.json'
STABILITY_PATH = ROOT / '.local' / 'eval_reports' / 'ocr_transcript_stability.json'

def _read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding='utf-8'))

review = _read_json(REVIEW_PATH)
all_cases_raw = _read_json(ALL_CASES_PATH)
stability = _read_json(STABILITY_PATH)

episodes = review.get('episodes', []) if isinstance(review, dict) else []
cases_all = all_cases_raw.get('cases', []) if isinstance(all_cases_raw, dict) else []
stability_cases = stability.get('cases', []) if isinstance(stability, dict) else []

episodes_df = pd.DataFrame(episodes)
cases_df = pd.DataFrame(stability_cases)
cases_all_df = pd.DataFrame(cases_all)

required_case_cols = ['id', 'lane', 'image_path', 'source_name']
for col in required_case_cols:
    if col not in cases_all_df.columns:
        cases_all_df[col] = None

if not cases_df.empty:
    cases_df = cases_df.merge(
        cases_all_df[required_case_cols],
        on='id',
        how='left',
        suffixes=('', '_from_cases'),
    )

if not cases_df.empty:
    cases_df['pass_runs'] = pd.to_numeric(cases_df.get('pass_runs', 0), errors='coerce').fillna(0).astype(int)
    cases_df['fail_runs'] = pd.to_numeric(cases_df.get('fail_runs', 0), errors='coerce').fillna(0).astype(int)
    cases_df['observed_runs'] = pd.to_numeric(cases_df.get('observed_runs', 0), errors='coerce').fillna(0).astype(int)
    cases_df['pass_rate'] = pd.to_numeric(cases_df.get('pass_rate', 0.0), errors='coerce').fillna(0.0)
    cases_df['binary_outcome'] = cases_df.apply(
        lambda r: 'pass' if int(r['pass_runs']) > int(r['fail_runs']) else 'fail', axis=1
    )
    cases_df['binary_value'] = cases_df['binary_outcome'].map({'pass': 1, 'fail': 0}).fillna(0).astype(int)

cases_summary = {
    'review_summary': review.get('summary', {}) if isinstance(review, dict) else {},
    'stability_summary': stability.get('summary', {}) if isinstance(stability, dict) else {},
    'counts': {
        'episodes': int(len(episodes_df)),
        'stability_cases': int(len(cases_df)),
        'all_cases': int(len(cases_all_df)),
    },
    'paths': {
        'review': str(REVIEW_PATH),
        'all_cases': str(ALL_CASES_PATH),
        'stability': str(STABILITY_PATH),
    },
}

cases_summary


In [ ]:
if cases_df.empty:
    print('No stability cases loaded yet. Run: make ocrall && make ocrstable')
else:
    lane_values = sorted(v for v in cases_df['lane'].dropna().astype(str).unique().tolist() if v)
    lane_filter = widgets.Dropdown(options=['all'] + lane_values, value='all', description='Lane')

    outcome_filter = widgets.ToggleButtons(
        options=['all', 'pass', 'fail'],
        value='all',
        description='Outcome',
    )

    min_pass_rate = widgets.FloatSlider(
        value=0.0, min=0.0, max=1.0, step=0.05, description='Min pass_rate'
    )

    max_runs = int(max(1, cases_df['observed_runs'].max()))
    min_runs = widgets.IntSlider(value=1, min=1, max=max_runs, step=1, description='Min runs')

    search_filter = widgets.Text(value='', description='Search', placeholder='id / source / path')

    limit_filter = widgets.IntSlider(value=40, min=10, max=200, step=10, description='Rows')

    sort_filter = widgets.Dropdown(
        options=['pass_rate', 'fail_runs', 'pass_runs', 'observed_runs', 'id'],
        value='pass_rate',
        description='Sort by',
    )

    desc_filter = widgets.Checkbox(value=True, description='Desc')

    controls = widgets.VBox(
        [
            widgets.HBox([lane_filter, outcome_filter]),
            widgets.HBox([min_pass_rate, min_runs]),
            widgets.HBox([search_filter, limit_filter]),
            widgets.HBox([sort_filter, desc_filter]),
        ]
    )

    output = widgets.Output()

    def _filtered() -> pd.DataFrame:
        df = cases_df.copy()
        if lane_filter.value != 'all':
            df = df[df['lane'].astype(str) == lane_filter.value]
        if outcome_filter.value != 'all':
            df = df[df['binary_outcome'] == outcome_filter.value]
        df = df[df['pass_rate'] >= float(min_pass_rate.value)]
        df = df[df['observed_runs'] >= int(min_runs.value)]

        query = search_filter.value.strip().lower()
        if query:
            haystack = (
                df['id'].astype(str).str.lower()
                + ' ' + df['source_name'].fillna('').astype(str).str.lower()
                + ' ' + df['image_path'].fillna('').astype(str).str.lower()
            )
            df = df[haystack.str.contains(query, na=False)]

        sort_col = sort_filter.value
        if sort_col in df.columns:
            df = df.sort_values(sort_col, ascending=not desc_filter.value, kind='stable')

        df = df.reset_index(drop=True)
        df['row_index'] = df.index + 1
        return df.head(int(limit_filter.value))

    def _render(*_args):
        df = _filtered()
        with output:
            clear_output(wait=True)
            if df.empty:
                print('No rows match current filters.')
                return

            fig = px.scatter(
                df,
                x='row_index',
                y='binary_value',
                color='binary_outcome',
                symbol='lane',
                hover_data={
                    'id': True,
                    'lane': True,
                    'pass_runs': True,
                    'fail_runs': True,
                    'pass_rate': ':.2f',
                    'observed_runs': True,
                    'source_name': True,
                    'image_path': True,
                    'binary_value': False,
                    'row_index': False,
                },
                color_discrete_map={'pass': '#2ca02c', 'fail': '#d62728'},
                title='OCR Eval Binary Outcome (filtered)',
                height=420,
            )
            fig.update_traces(marker={'size': 10, 'line': {'width': 1, 'color': '#111'}})
            fig.update_yaxes(tickvals=[0, 1], ticktext=['FAIL', 'PASS'], range=[-0.2, 1.2])
            fig.update_xaxes(title='Filtered case index')
            fig.update_layout(legend_title='Outcome', template='plotly_white')
            fig.show()

            cols = [
                'id', 'lane', 'binary_outcome', 'pass_runs', 'fail_runs',
                'pass_rate', 'observed_runs', 'source_name', 'image_path'
            ]
            existing_cols = [c for c in cols if c in df.columns]
            display(df[existing_cols])

            if 'image_path' in df.columns:
                links = df[['id', 'image_path']].copy()
                links['image_path'] = links['image_path'].fillna('').astype(str).apply(
                    lambda p: f'<a href="file://{p}" target="_blank">{p}</a>' if p else ''
                )
                display(HTML('<h4>Asset paths</h4>' + links.to_html(escape=False, index=False)))

    for widget in [
        lane_filter, outcome_filter, min_pass_rate, min_runs,
        search_filter, limit_filter, sort_filter, desc_filter,
    ]:
        widget.observe(_render, names='value')

    display(controls, output)
    _render()


In [ ]:
# Quick lane summary table
if not cases_df.empty:
    lane_summary = (
        cases_df.groupby('lane', dropna=False)
        .agg(
            total=('id', 'count'),
            pass_cases=('binary_outcome', lambda s: int((s == 'pass').sum())),
            fail_cases=('binary_outcome', lambda s: int((s == 'fail').sum())),
            avg_pass_rate=('pass_rate', 'mean'),
        )
        .reset_index()
        .sort_values('total', ascending=False)
    )
    lane_summary['avg_pass_rate'] = lane_summary['avg_pass_rate'].round(3)
    lane_summary
else:
    print('No stability cases available yet.')
